## powershell to open a matlab file in the desktop version of matlab
To open a MATLAB file using PowerShell in the desktop version of MATLAB, you can use the following command:

```powershell
!matlab -desktop -r "open('C:\path\to\your\file.m');"
```

### Flags
-desktop flag - This flag tells MATLAB to open in desktop mode, which provides the full graphical user interface.
-r flag - This flag allows you to run a command or script immediately after MATLAB starts. In this case, it runs the `open` function to open the specified MATLAB file.

In [ ]:
import hdf5storage
import numpy as np
import re
import pandas as pd
from scipy.io.matlab import mat_struct, MatlabOpaque
import os
from dotenv import load_dotenv

load_dotenv()
DIR = os.getenv("DIR")

raw = hdf5storage.loadmat(DIR)

In [ ]:
def _decode(x):
    return x.decode() if isinstance(x, bytes) else x

def _describe_opaque(obj):
    """MATLAB `string` arrays / classdef objects are stored as MCOS references
    that neither scipy nor hdf5storage can decode into real data -- this is a
    reader limitation, not something fixable in Python post-hoc. Resave the
    offending variable in MATLAB as char/cellstr (e.g. cellstr(x)) if you need
    its actual contents."""
    rec = obj[0]
    names = obj.dtype.names or ()
    if "_Class" in names:  # hdf5storage's opaque-object encoding
        return {
            "_matlab_unsupported_class": _decode(rec["_Class"]),
            "_matlab_type_system": _decode(rec["_TypeSystem"]) if "_TypeSystem" in names else None,
        }
    if "s2" in names:  # scipy's opaque-object encoding
        return {
            "_matlab_unsupported_class": _decode(rec["s2"]),
            "_matlab_variable_name": _decode(rec["s0"]) if "s0" in names else None,
        }
    return {"_matlab_unsupported_fields": list(names)}

def mat_to_py(obj):
    """Recursively normalize MATLAB struct/cell data loaded via EITHER
    scipy.io.loadmat(struct_as_record=False, squeeze_me=True) OR
    hdf5storage.loadmat into plain dicts/lists/scalars/ndarrays, so the rest of
    the pipeline (frame_structs below) doesn't care which loader produced it."""
    if isinstance(obj, mat_struct):  # scipy struct
        return {name: mat_to_py(getattr(obj, name)) for name in obj._fieldnames}

    if isinstance(obj, MatlabOpaque):
        return _describe_opaque(obj)

    if isinstance(obj, np.void):  # a single hdf5storage struct-array record
        return {name: mat_to_py(obj[name]) for name in obj.dtype.names}

    if isinstance(obj, np.ndarray):
        if obj.dtype.names:  # hdf5storage struct array
            items = [mat_to_py(rec) for rec in obj.ravel()]
            return items[0] if len(items) == 1 else items
        if obj.dtype == object:  # cell array
            squeezed = np.squeeze(obj)
            if squeezed.ndim == 0:
                return mat_to_py(squeezed.item())
            return [mat_to_py(x) for x in squeezed.ravel()]
        squeezed = np.squeeze(obj)
        return squeezed.item() if squeezed.ndim == 0 else squeezed

    return obj

data = {k: mat_to_py(v) for k, v in raw.items() if not k.startswith("__")}
data.keys()
def _safe_name(s):
    """Turn an arbitrary string into a valid Python identifier."""
    name = re.sub(r"[^0-9a-zA-Z_]", "_", s)
    name = re.sub(r"_+", "_", name).strip("_")
    if not name:
        name = "_"
    if name[0].isdigit():
        name = "_" + name
    return name

_name_counts = {}
def _assign_name(path):
    """Name a variable after only the bottom-level (leaf) field name in its
    struct path, e.g. 'emgMatrix.hierarchy.emgBinNames' -> 'emgBinNames'.
    Since MATLAB reuses field names at different nesting depths (e.g.
    'perturbationLevel' appears once per perturbationDirection row), repeats
    get a numeric suffix (_2, _3, ...) so nothing is silently overwritten."""
    leaf = _safe_name(path.rsplit(".", 1)[-1])
    n = _name_counts.get(leaf, 0) + 1
    _name_counts[leaf] = n
    return leaf if n == 1 else f"{leaf}_{n}"

mat_variable_names = []

def flatten_structs(obj, path):
    """Recursively hoist every MATLAB struct / struct-array into its own
    DataFrame saved directly in the environment (named by _assign_name),
    instead of nesting a DataFrame inside another DataFrame's cell. Returns
    (hoisted, result): if hoisted, result is the variable name it was saved
    as (used to build a breadcrumb in the parent, e.g. "-> emgMatrix"); if
    not hoisted, result is the original value (plain array/list/scalar),
    left as a normal cell value. Makes no assumption about field names, so it
    works on any .mat file loaded the same way."""
    if isinstance(obj, dict):
        row = {}
        for k, v in obj.items():
            child_path = f"{path}.{k}"
            hoisted, result = flatten_structs(v, child_path)
            row[k] = f"-> {result}" if hoisted else result
        varname = _assign_name(path)
        globals()[varname] = pd.DataFrame([row])
        mat_variable_names.append(varname)
        return True, varname
    if isinstance(obj, list) and obj and all(isinstance(x, dict) for x in obj):
        keys = obj[0].keys()
        if all(x.keys() == keys for x in obj):
            rows = []
            for i, item in enumerate(obj):
                row = {}
                for k, v in item.items():
                    child_path = f"{path}[{i}].{k}"
                    hoisted, result = flatten_structs(v, child_path)
                    row[k] = f"-> {result}" if hoisted else result
                rows.append(row)
            varname = _assign_name(path)
            globals()[varname] = pd.DataFrame(rows)
            mat_variable_names.append(varname)
            return True, varname
    return False, obj

for _name, _value in data.items():
    _hoisted, _result = flatten_structs(_value, _name)
    if not _hoisted:
        # top-level value wasn't a struct/struct-array (plain matrix, vector,
        # cell array, or scalar) -- still give it its own DataFrame so every
        # variable in the file ends up saved the same way
        if isinstance(_result, np.ndarray):
            _df = pd.DataFrame(_result) if _result.ndim > 1 else pd.DataFrame({"value": _result})
        elif isinstance(_result, list):
            _df = pd.DataFrame({"value": _result})
        else:
            _df = pd.DataFrame({"value": [_result]})
        _varname = _assign_name(_name)
        globals()[_varname] = _df
        mat_variable_names.append(_varname)

# the raw loader output and every intermediate variable used to build these
# are no longer needed now that each table is its own variable
del raw, data, _name, _value, _hoisted, _result, _name_counts
if "_df" in globals():
    del _df, _varname

print(len(mat_variable_names), "DataFrames saved to the environment:")
for _n in mat_variable_names:
    _df = globals()[_n]
    print(f"{_n}: {_df.shape} cols={list(_df.columns)[:6]}{'...' if len(_df.columns) > 6 else ''}")
del _n, _df
def drop_mat_variables(keep=()):
    """Delete the DataFrames created above from the environment, except the
    ones named in `keep`. Updates mat_variable_names in place to reflect
    what's left, so this is safe to call more than once (e.g. to narrow the
    keep-list further on a second pass)."""
    keep = set(keep)
    kept = []
    for name in mat_variable_names:
        if name in keep:
            kept.append(name)
        elif name in globals():
            del globals()[name]
    mat_variable_names[:] = kept
    print(f"{len(mat_variable_names)} variable(s) remain: {mat_variable_names}")
